# ACDC cardiac MRI segmentation -- evaluation & figures

Loads the checkpoints and `results/all_results.json` produced by
`01_train.ipynb`, then:

1. measures **per-slice** (batch_size=1) inference latency for each encoder
2. computes per-class Dice (LV cavity / RV cavity / Myocardium)
3. generates the qualitative segmentation comparison figure
4. generates the metrics bar chart (Dice, mIoU, latency, model size)

Everything reads from and writes back to `results/all_results.json` --
**no numbers are hardcoded in this notebook.** If you re-train and get
different numbers, re-running this notebook is all you need to do; every
table and figure regenerates from the updated file.


## Setup

In [ ]:
%pip install -q lightning nibabel segmentation-models-pytorch kagglehub

import sys
sys.path.insert(0, "..")

import json
import os

import matplotlib.pyplot as plt
import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader

from src.dataset import ACDC2DDataModule, CLASS_NAMES
from src.model import LitUNet2D
from src.losses import dice_score, per_class_dice
from src.benchmark import measure_per_slice_latency, load_results, save_results, update_result
from src.visualize import find_best_slice, plot_segmentation_comparison, plot_metrics_comparison


In [ ]:
ENCODERS = ["resnet18", "vgg16", "mobilenet_v2", "resnet50"]
FOREGROUND_CLASS_NAMES = {1: "LV", 2: "RV", 3: "Myocardium"}

CHECKPOINT_DIR = "../checkpoints"
RESULTS_PATH = "../results/all_results.json"
FIGURES_DIR = "../figures"

os.makedirs(FIGURES_DIR, exist_ok=True)

results = load_results(RESULTS_PATH)
assert results, f"No results found at {RESULTS_PATH} -- run 01_train.ipynb first."
print(json.dumps(results, indent=2))


## Data

Re-creates the same train/val/test split as training (same `seed`), so the
test set here is identical to the held-out set the models were scored on
during training.


In [ ]:
import kagglehub

data_path = kagglehub.dataset_download("samdazel/automated-cardiac-diagnosis-challenge-miccai17")
acdc_root = f"{data_path}/database/training"

pl.seed_everything(42, workers=True)
dm = ACDC2DDataModule(acdc_root, batch_size=8, num_workers=2, seed=42)
dm.setup()

print(f"test slices: {len(dm.test_ds)}")


In [ ]:
def load_trained_model(encoder, device="cuda"):
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"{encoder}_best.pth")
    model = LitUNet2D(lr=1e-3, encoder_name=encoder)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model = model.to(device)
    model.eval()
    return model


## 1. Per-slice inference latency

Single-sample (batch_size=1) timing using CUDA events, which is the
realistic per-image latency a deployed model would see -- as opposed to the
batch-8 latency measured during training, which is throughput-oriented and
not representative of single-image inference time.

This updates `results/all_results.json` in place, adding
`inference_ms_per_slice_mean/median/std` next to the existing
`inference_ms_batch8_mean` from training -- both are kept, clearly labeled,
so nothing is overwritten or ambiguous.


In [ ]:
print(f"{'Encoder':<18} {'Mean (ms)':>12} {'Median (ms)':>12} {'Std (ms)':>10}")
print("-" * 56)

for encoder in ENCODERS:
    model = load_trained_model(encoder)
    latency = measure_per_slice_latency(model, dm.test_ds, device="cuda", n_warmup=20, n_measure=100)

    update_result(
        results, encoder,
        inference_ms_per_slice_mean=latency["mean"],
        inference_ms_per_slice_median=latency["median"],
        inference_ms_per_slice_std=latency["std"],
    )

    print(f"{encoder:<18} {latency['mean']:>12} {latency['median']:>12} {latency['std']:>10}")

save_results(results, RESULTS_PATH)
print(f"\nSaved to {RESULTS_PATH}")


## 2. Per-class Dice (LV / RV / Myocardium)

In [ ]:
test_loader = DataLoader(dm.test_ds, batch_size=1, shuffle=False, num_workers=0)

print(f"{'Encoder':<18} {'LV':>8} {'RV':>8} {'Myocardium':>12}")
print("-" * 50)

per_class_results = {}

for encoder in ENCODERS:
    model = load_trained_model(encoder)

    all_preds, all_targets = [], []
    with torch.no_grad():
        for x, y in test_loader:
            pred = torch.argmax(model(x.to("cuda")), dim=1).squeeze().cpu()
            all_preds.append(pred)
            all_targets.append(y.squeeze())

    preds = torch.stack(all_preds)
    targets = torch.stack(all_targets)

    scores = per_class_dice(preds, targets, FOREGROUND_CLASS_NAMES)
    per_class_results[encoder] = scores
    print(f"{encoder:<18} {scores['LV']:>8} {scores['RV']:>8} {scores['Myocardium']:>12}")

with open("../results/per_class_dice.json", "w") as f:
    json.dump(per_class_results, f, indent=2)
print("\nSaved to ../results/per_class_dice.json")


## 3. Qualitative segmentation comparison

Picks a representative test slice that contains all three foreground
classes (LV, RV, myocardium), then shows input / ground truth / prediction
for every encoder.


In [ ]:
best_idx = find_best_slice(dm.test_ds, target_classes=(1, 2, 3))
print(f"Using test slice index {best_idx} (contains LV, RV, and myocardium)")

sample = dm.test_ds[best_idx]


In [ ]:
models_by_encoder = {enc: load_trained_model(enc) for enc in ENCODERS}
dice_by_encoder = {enc: results[enc]["dice"] for enc in ENCODERS}

fig = plot_segmentation_comparison(
    models_by_encoder, sample, dice_by_encoder, ENCODERS,
    device="cuda", save_path=f"{FIGURES_DIR}/segmentation_comparison.png",
)
plt.show()
print("Saved figures/segmentation_comparison.png")


## 4. Metrics bar chart

Dice, mIoU, latency, and model size side by side. `latency_key` controls
which latency number is plotted -- defaults to the per-slice mean computed
above (the realistic single-image latency), not the batch-8 throughput
number from training.


In [ ]:
fig = plot_metrics_comparison(
    results, ENCODERS,
    latency_key="inference_ms_per_slice_mean",
    save_path=f"{FIGURES_DIR}/metrics_comparison.png",
)
plt.show()
print("Saved figures/metrics_comparison.png")


## Summary

`results/all_results.json` now contains, for every encoder: test-set Dice
and mIoU, model size, batch-8 latency (from training), and per-slice
latency mean/median/std (measured above). `results/per_class_dice.json`
has the LV/RV/Myocardium breakdown. Figures are saved under `figures/`.
